# Compare spike type firing between episodes

In [13]:
raw_dir = 'e:/Data/BogiSejtek/20251007/data/invivo/'
feature_dir = 'e:/tmp/RaApTemp/Preprocessed/SpikeFeatures/'
results_dir = 'e:/tmp/RaApTemp/Preprocessed/EpisodeFiring/'

Set up result directories.

In [14]:
from pathlib import Path
result_save_path = Path(results_dir) / 'GroupComparison' / 'Results'
result_save_path.mkdir(parents=True, exist_ok=True)
figure_save_path = Path(results_dir) / 'GroupComparison' / 'Figures'
figure_save_path.mkdir(parents=True, exist_ok=True)

In [15]:
input_config = {}
input_config['table_path'] = 'e:/tmp/RaApTemp/Preprocessed/data_invivo.xlsx'
input_config['select_rules'] = [{'selected': 1}]

In [16]:
spike_types = ['AIS-AP', 'UNKNOWN', 'RA-AP']
episodes = ['sw', 'theta', 'awake']
aggregation_name = 'mean'  # or 'median'
result_suffix = '_episode_firing_rate_by_spiketype'
dpi = 100
make_svg = True

Load the table with the cell list, selected by the selector table.

In [17]:
from workflows.run_util.io import get_cell_info_list
file_list = get_cell_info_list(input_config)
print(f'Found {len(file_list)} cells matching selection criteria.')

Found 17 cells matching selection criteria.


Load the selected input files from the given path.

In [18]:
import pandas as pd
from common.data_utils import CellWrapper

cell_frames = []
for info in file_list:
    feature_path = (
        Path(results_dir) /
        'Individual' /
        info.cell_type /
        f'{info.name}{result_suffix}.xlsx'
    )
    cell_df = pd.read_excel(feature_path)
    cell_frames.append(CellWrapper(info, cell_df))
print(f'Loaded data from {len(cell_frames)} cells.')

Loaded data from 17 cells.


Aggregate each cell's data.

In [19]:
import numpy as np

aggregate = np.nanmean if aggregation_name == 'mean' else np.nanmedian
cellwise_aggregates = []
for cell in cell_frames:
    frame = cell.data
    cell_episodes = []
    for episode in episodes:
        subset = frame[frame['episode'] == episode]
        episode_result = {
            'name': cell.info.name,
            'group': cell.info.cell_type,
            'episode': episode
        }
        aggregation = {
            spike_type: aggregate(subset[spike_type]) if subset.shape[0] > 0 else np.nan
            for spike_type in spike_types
        }
        cell_episodes.append(episode_result | aggregation)
    cellwise_aggregates.append(pd.DataFrame(cell_episodes))
overall_results = pd.concat(cellwise_aggregates)

Save the cellwise aggregated data.

In [20]:
overall_results.to_excel(
    result_save_path / f'cellwise_aggregate.xlsx', index=False
)
overall_results['episode'] = pd.Categorical(
    overall_results['episode'], categories=episodes, ordered=True
)

Plot the results and save the figures.

In [21]:
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import wilcoxon, kruskal

for group in overall_results['group'].unique():
    group_df = overall_results[overall_results['group'] == group]
    fig, axes = plt.subplots(1, 3, figsize=(8, 6), sharey=True)
    for i, spike_type in enumerate(spike_types):
        ax = axes[i]
        sns.boxplot(
            x='episode',
            y=spike_type,
            fill=False,
            fliersize=0,
            color='#808080',
            data=group_df[~group_df[spike_type].isna()],
            ax=ax
        )
        if len(episodes) == 2:
            res = wilcoxon(
                group_df[group_df['episode'] == episodes[0]][spike_type],
                group_df[group_df['episode'] == episodes[1]][spike_type]
            )
            stat_result = (
                f"{episodes[0]} vs. {episodes[1]}, " +
                f"p={res.pvalue:.3f}"  # type: ignore
            )
        else:
            _, kruskal_p = kruskal(
                *[
                    group_df[group_df['episode'] == ep][spike_type].dropna()
                    for ep in episodes
                ]
            )
            stat_result = f"Kruskal-Wallis p={kruskal_p:.3f}"

        res = wilcoxon(
            overall_results[overall_results['episode'] == 'sw'][spike_type],
            overall_results[overall_results['episode'] == 'theta'][spike_type]
        )
        for name in group_df['name'].unique():
            subset = group_df[group_df['name'] == name]
            ax.plot(
                subset['episode'],
                subset[spike_type],
                markersize=10,
                marker='o',
                linestyle='-',
                label=name,
                linewidth=2,
            )

        ax.set_title(f"{spike_type} - {stat_result}") # type: ignore
        ax.set_ylabel("Mean firing rate (Hz)")

    plt.suptitle(f"{group} cells")
    plt.tight_layout()

    figure_name = f'{group}_episode_firing_comparison'
    fig.savefig(
        figure_save_path / f'{figure_name}.png',
        dpi=dpi,
        format='png'
    )
    if make_svg:
        fig.savefig(
            figure_save_path / f'{figure_name}.svg',
            format='svg'
        )
    plt.close(fig)


C:\Users\User\AppData\Local\Temp\ipykernel_11340\3840273816.py:30: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  _, kruskal_p = kruskal(
C:\Users\User\AppData\Local\Temp\ipykernel_11340\3840273816.py:30: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  _, kruskal_p = kruskal(
C:\Users\User\AppData\Local\Temp\ipykernel_11340\3840273816.py:30: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  _, kruskal_p = kruskal(
C:\Users\User\AppData\Local\Temp\ipykernel_11340\3840273816.py:30: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  _, kruskal_p = kruskal(
C:\Users\User\AppData\Local\Temp\ipykernel_11340\384